# SKANN-SSL V3.2.0 Training Notebook (HYBRID)

**Version:** V3.2.0 HYBRID  
**Date:** January 2026  
**Platform:** Kaggle Dual T4 GPUs (DDP)

---

## 🔀 HYBRID STRATEGY: Curriculum Learning

| Phase | Epochs | Strategy | Positive Selection |
|-------|--------|----------|-------------------|
| 1 | **1-25** | **SupCon** | Random clip, same class |
| 2 | **26-50** | **SSL** | Hard positive K=6 (vessels) / K=3 (no_vessel) |

**Rationale:**
- SupCon first → Model learns broad class boundaries
- SSL second → Model refines with hard positives for invariance

---

## V3.2 Changes from V3.1

| Parameter | V3.1 (failed) | V3.2 (this) | Reason |
|-----------|---------------|-------------|--------|
| Total Clips | 2,400 | **12,000** | 5× more data |
| Clips/Class | 480 | **2,400** | Balanced |
| Epochs | 50 | **50** | Same as V2 baseline |
| Physical Batch | 2 | 2 | Memory constraint (5s clips) |
| Grad Accumulation | 1 | **8** | Effective batch = 32 |
| Warmup Epochs | 0 | **3** | Stabilize early training |
| Physics Bugs | ❌ Multiple | **✅ Fixed** | 2Hz artifact resolved |
| **Pairing** | N/A | **Hybrid** | SupCon (1-25) → SSL (26-50) |

### Preserved from V2.1.0 (baseline)
- SK kernel sizes: (31, 63, 127, 255, 511, 1023)
- Projector: 512 → 4096 → 8192 → 128
- Barlow Twins loss (λ=5e-3)
- SyncBatchNorm for DDP stability
- Learning rate: 1e-4

---

## Pipeline

0. Requirements / Compatibility
1. Version Check
2. Environment Setup
3. Data Ingestion & Validation
4. Training Script (embedded)
5. Launch Training
6. Extract Embeddings
7. Silhouette Analysis
8. Silhouette Distribution Plot
9. UMAP Visualization
10. Export Production Bundle
11. Summary & Next Steps


---
## Cell 0: Requirements / Compatibility

In [ ]:
# Cell 0: Requirements / compatibility
import sys, subprocess, importlib.util

def _pkg_version(pkg: str):
    try:
        mod = __import__(pkg)
        return getattr(mod, '__version__', None)
    except Exception:
        return None

REQUIRED = {
    'scikit-learn': '1.2',
    'umap-learn': '0.5.5',
    'joblib': '1.2',
}

def _ver_tuple(v: str):
    parts = (v or '').split('.')
    if len(parts) < 2:
        return (0, 0)
    try:
        return (int(parts[0]), int(parts[1]))
    except Exception:
        return (0, 0)

needs_install = False
for pkg, min_ver in REQUIRED.items():
    name = pkg.replace('-', '_')
    try:
        spec = importlib.util.find_spec(name)
        if spec is None:
            needs_install = True
            break
        ver = _pkg_version(name)
        if ver is None or _ver_tuple(ver) < _ver_tuple(min_ver):
            needs_install = True
            break
    except Exception:
        needs_install = True
        break

if needs_install:
    print('📦 Installing/upgrading required packages…')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'scikit-learn>=1.2',
        'umap-learn>=0.5.5',
        'joblib>=1.2',
    ])
    print('✅ Packages installed.')
else:
    print('✅ Required packages already satisfied')

try:
    import sklearn, umap, joblib
    print('sklearn:', sklearn.__version__)
    print('umap-learn:', umap.__version__)
    print('joblib:', joblib.__version__)
except Exception as e:
    print('⚠️ Version check skipped:', e)

---
## Cell 1: Version Check

In [ ]:
# Cell 1: Version Check
print("="*60)
print("  SKANN-SSL V3.2.0 — 12K Clips, 5 Classes, Physics Fixed")
print("="*60)

try:
    import torch
    import numpy as np
    import pandas as pd
    import sklearn
    import umap
    import joblib
    print(f'PyTorch: {torch.__version__}')
    print(f'NumPy: {np.__version__}')
    print(f'Pandas: {pd.__version__}')
    print(f'Scikit-learn: {sklearn.__version__}')
    print(f'UMAP: {umap.__version__}')
    print(f'Joblib: {joblib.__version__}')
    print(f'CUDA Available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU Count: {torch.cuda.device_count()}')
        for i in range(torch.cuda.device_count()):
            print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
except Exception as e:
    print(f'⚠️ Version check error: {e}')

---
## Cell 2: Environment Setup

In [ ]:
# Cell 2: Environment Setup
import os
import gc
import sys
import time
import datetime
import traceback

def setup_environment():
    print("🧹 Environment Setup...")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = '29500'
    os.makedirs("/kaggle/working/run_logs", exist_ok=True)
    gc.collect()
    print("✅ Environment configured.")

def safe_cleanup():
    import torch
    print("🧹 Gentle cleanup...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    print("✅ Cleanup done.")

def log(msg, rank=None):
    ts = time.strftime('%H:%M:%S')
    if rank is not None:
        print(f"[{ts}][R{rank}] {msg}", flush=True)
    else:
        print(f"[{ts}] {msg}", flush=True)

setup_environment()
safe_cleanup()
log("✅ All utilities loaded and environment ready")

---
## Cell 3: Data Ingestion & Validation

In [ ]:
# Cell 3: Data Ingestion & Validation
import os
import pandas as pd
import numpy as np

# ============================================================================
# V3.2 DATASET CONFIGURATION
# ============================================================================
V3_DATASET_PATH = "/kaggle/input/skann-ssl-v3-2-dataset"
TENSOR_DIR = f"{V3_DATASET_PATH}/tensors"
MANIFEST_PATH = f"{V3_DATASET_PATH}/master_dataset_manifest.csv"
PAIRING_PATH = f"{V3_DATASET_PATH}/pairing_manifest.csv"

# V3.2 Audio Parameters
SAMPLE_RATE = 16000
DURATION_SEC = 5.0
SAMPLES_PER_CLIP = int(SAMPLE_RATE * DURATION_SEC)  # 80,000
NUM_CLASSES = 5
CLASS_NAMES = ['cargo_ship', 'fishing_vessel', 'no_vessel', 'small_craft', 'tanker']

print("🔍 SCANNING FOR V3.2 DATASET...")
print(f"   Expected path: {V3_DATASET_PATH}")

# Validate dataset exists
if not os.path.exists(V3_DATASET_PATH):
    raise FileNotFoundError(f"V3.2 dataset not found at {V3_DATASET_PATH}")

print(f"✅ FOUND V3.2 DATASET")

# Load manifest
manifest_df = pd.read_csv(MANIFEST_PATH)

print("\n" + "="*50)
print("📊 V3.2 DATASET STATISTICS")
print("="*50)
print(f"Total Clips:          {len(manifest_df)}")
print(f"Classes:              {sorted(manifest_df['vessel_class'].unique())}")
print(f"Samples/Clip:         {SAMPLES_PER_CLIP}")
print(f"Duration:             {DURATION_SEC}s")

# Class distribution
print("\n📈 Class Distribution:")
for cls in sorted(manifest_df['vessel_class'].unique()):
    count = len(manifest_df[manifest_df['vessel_class'] == cls])
    print(f"   {cls}: {count} clips")

# Verify tensor shape
sample_tensor = np.load(os.path.join(TENSOR_DIR, "tensor_000000.npy"))
print(f"\n📐 Sample tensor shape: {sample_tensor.shape}")

# Count tensor files
tensor_files = [f for f in os.listdir(TENSOR_DIR) if f.endswith('.npy')]
print(f"📁 Tensor files found: {len(tensor_files)}")

# Check pairing manifest exists
if not os.path.exists(PAIRING_PATH):
    print(f"❌ ERROR: pairing_manifest.csv not found at {PAIRING_PATH}")
    print("   Run generate_pairing_manifest_v3_2.py first!")
    raise FileNotFoundError("pairing_manifest.csv missing")
else:
    pairing_df = pd.read_csv(PAIRING_PATH)
    print(f"\n✅ Pairing manifest found: {len(pairing_df)} anchors")
    
    # Show K per class
    print("   Partners per class:")
    for cls in sorted(pairing_df['vessel_class'].unique()):
        grp = pairing_df[pairing_df['vessel_class'] == cls]
        avg_k = grp['partner_clip_ids'].apply(lambda x: len(str(x).split('|'))).mean()
        print(f"      {cls}: K={avg_k:.0f}")

# V3.2 Specific: Physics validation
print("\n" + "="*50)
print("🔬 V3.2 PHYSICS VALIDATION")
print("="*50)
vessel_df = manifest_df[manifest_df['vessel_class'] != 'no_vessel']
res1_ok = (vessel_df['resonance_freq_1'] > 0).sum()
res2_ok = (vessel_df['resonance_freq_2'] > 0).sum()
print(f"   res_1 coverage: {res1_ok}/{len(vessel_df)} ({100*res1_ok/len(vessel_df):.1f}%)")
print(f"   res_2 coverage: {res2_ok}/{len(vessel_df)} ({100*res2_ok/len(vessel_df):.1f}%)")
print(f"   2Hz artifact:   ✅ Fixed (swell_freq randomized)")

print("\n🚀 V3.2 DATASET IS READY FOR TRAINING!")


---
## Cell 4: Training Script (Embedded)

V3.2 Changes:
- Gradient accumulation (8 steps → effective batch 32)
- Learning rate warmup (3 epochs)
- 50 epochs (same as V2 baseline)

In [ ]:
%%writefile train_script_v3_2.py
"""
SKANN-SSL V3.2.0 HYBRID Training Script

Key Changes from V3.1:
- 12,000 clips (5× more data)
- Gradient accumulation: 8 steps
- Learning rate warmup: 3 epochs
- Epochs: 50 (same as V2 baseline)
- Physics bugs fixed (2Hz artifact resolved)

Preserved from V2.1.0:
- SK kernel sizes: (31, 63, 127, 255, 511, 1023)
- Projector: 512 → 4096 → 8192 → 128
- Barlow Twins loss (λ=5e-3)
"""

import os
import time
import traceback
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP

# ============================================================================
# V3.2 CONFIGURATION
# ============================================================================
SAMPLES_PER_CLIP = 80000  # 5 seconds @ 16kHz
NUM_CLASSES = 5
LATENT_DIM = 128
SK_KERNEL_SIZES = (31, 63, 127, 255, 511, 1023)

# V3.2 Training hyperparameters
EPOCHS = 50              # Same as V2 baseline
BATCH_SIZE = 2           # Physical batch per GPU
GRAD_ACCUM_STEPS = 8     # Effective batch = 2 * 2 GPUs * 8 = 32
BASE_LR = 1e-4
WARMUP_EPOCHS = 3
WEIGHT_DECAY = 0.01
BT_LAMBDA = 5e-3         # Barlow Twins redundancy reduction


# ============================================================================
# LOGGING UTILITIES
# ============================================================================

class RankLogger:
    def __init__(self, rank):
        self.rank = rank
    def log(self, msg):
        ts = time.strftime('%H:%M:%S')
        print(f"[{ts}][R{self.rank}] {msg}", flush=True)

def write_heartbeat(log_dir, epoch, step, loss, rank):
    path = os.path.join(log_dir, f"heartbeat_r{rank}.txt")
    with open(path, 'w') as f:
        f.write(f"{time.time()},{epoch},{step},{loss:.4f}\n")

def log_crash(log_dir, exc, rank):
    path = os.path.join(log_dir, f"crash_r{rank}.txt")
    with open(path, 'w') as f:
        f.write(f"Rank {rank} crashed at {time.strftime('%H:%M:%S')}\n")
        f.write(traceback.format_exc())
    print(f"[CRASH] Rank {rank}: {exc}")


# ============================================================================
# LEARNING RATE SCHEDULER WITH WARMUP
# ============================================================================

class WarmupCosineScheduler:
    """Linear warmup followed by cosine annealing."""
    
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.current_epoch = 0
    
    def step(self):
        self.current_epoch += 1
        if self.current_epoch <= self.warmup_epochs:
            # Linear warmup
            lr = self.base_lr * (self.current_epoch / self.warmup_epochs)
        else:
            # Cosine annealing
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.base_lr * 0.5 * (1 + math.cos(math.pi * progress))
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        return lr
    
    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']


# ============================================================================
# SELECTIVE KERNEL FILTERBANK (V2.1.0 Architecture)
# ============================================================================

class SKFilterbank(nn.Module):
    """
    Multi-branch Selective Kernel filterbank with attention-weighted fusion.
    
    Kernel sizes chosen for underwater acoustic physics:
    - k=31:   captures cavitation (500+ Hz)
    - k=63:   captures resonance (250+ Hz)
    - k=127:  captures blade pass (125+ Hz)
    - k=255:  captures generator 50Hz (62+ Hz)
    - k=511:  captures generator 25Hz (31+ Hz)
    - k=1023: captures shaft rate (15+ Hz)
    """
    
    def __init__(self, in_channels=1, out_channels=64, kernel_sizes=SK_KERNEL_SIZES):
        super().__init__()
        self.n_branches = len(kernel_sizes)
        
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels, k, padding=k//2, bias=False),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=False)
            )
            for k in kernel_sizes
        ])
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(out_channels, out_channels // 4)
        self.attn = nn.Linear(out_channels // 4, self.n_branches)
    
    def forward(self, x):
        feats = [branch(x) for branch in self.branches]
        stacked = torch.stack(feats, dim=1)
        
        fused = stacked.sum(dim=1)
        gap = self.gap(fused).squeeze(-1)
        attn_weights = torch.softmax(self.attn(F.relu(self.fc(gap))), dim=1)
        
        attn_weights = attn_weights.unsqueeze(-1).unsqueeze(-1)
        out = (stacked * attn_weights).sum(dim=1)
        
        return out


# ============================================================================
# HYBRID SK ENCODER V3.2
# ============================================================================

class HybridSKEncoderV3(nn.Module):
    """
    V3.2 encoder for 5-second clips (80,000 samples).
    
    Architecture:
    - SKFilterbank: [B, 1, 80000] → [B, 64, 80000]
    - Conv blocks with downsampling
    - Global pooling → 512-dim
    - Projector → 128-dim latent
    """
    
    def __init__(self, latent_dim=128):
        super().__init__()
        
        self.sk_filterbank = SKFilterbank(in_channels=1, out_channels=64)
        
        self.conv_blocks = nn.Sequential(
            # Block 1: 64 → 128, downsample 4x
            nn.Conv1d(64, 128, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=False),
            
            # Block 2: 128 → 256, downsample 4x
            nn.Conv1d(128, 256, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            
            # Block 3: 256 → 512, downsample 4x
            nn.Conv1d(256, 512, kernel_size=7, stride=4, padding=3, bias=False),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=False),
            
            # Block 4: 512 → 512, downsample 5x (for 80k samples)
            nn.Conv1d(512, 512, kernel_size=5, stride=5, padding=2, bias=False),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=False),
        )
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.projector = nn.Sequential(
            nn.Linear(512, 4096),
            nn.BatchNorm1d(4096),
            nn.ReLU(inplace=False),
            nn.Linear(4096, 8192),
            nn.BatchNorm1d(8192),
            nn.ReLU(inplace=False),
            nn.Linear(8192, latent_dim)
        )
    
    def forward(self, x):
        if x.dim() == 4:
            x = x.squeeze(1)
        
        x = self.sk_filterbank(x)
        x = self.conv_blocks(x)
        x = self.global_pool(x).squeeze(-1)
        x = self.projector(x)
        
        return x


# ============================================================================
# DATASET
# ============================================================================

class HybridDataset(Dataset):
    """
    HYBRID APPROACH: Curriculum learning with SupCon → SSL transition.
    
    Strategy (TEMPORAL SPLIT):
    - Epochs 1-25:  SupCon - Random clip from same class (broad clustering)
    - Epochs 26-50: SSL - Hard positive from pairing manifest K=6/K=3 (refined invariance)
    
    The current_epoch is set externally before each epoch via set_epoch().
    """
    
    SUPCON_EPOCHS = 25  # First 25 epochs use SupCon
    
    def __init__(self, manifest_path, pairing_path, data_dir):
        self.data_dir = data_dir
        self.current_epoch = 1  # Will be updated by training loop
        
        # Load manifests
        self.df = pd.read_csv(manifest_path)
        pairing_df = pd.read_csv(pairing_path)
        
        # Build class mapping
        self.classes = sorted(self.df['vessel_class'].unique())
        self.class_to_id = {c: i for i, c in enumerate(self.classes)}
        
        # Build class-to-clip_ids mapping for SupCon sampling
        self.class_clip_ids = {}
        for cls in self.classes:
            self.class_clip_ids[cls] = self.df[self.df['vessel_class'] == cls]['clip_id'].tolist()
        
        # Build pairing lookup for SSL sampling (clip_id -> partner_clip_ids list)
        self.pairing_lookup = {}
        for _, row in pairing_df.iterrows():
            anchor_id = int(row['anchor_clip_id'])
            partner_ids = [int(x) for x in str(row['partner_clip_ids']).split('|')]
            self.pairing_lookup[anchor_id] = partner_ids
        
        print(f"[HybridDataset] Loaded {len(self.df)} clips")
        print(f"   Epochs 1-{self.SUPCON_EPOCHS}: SupCon (dynamic sampling)")
        print(f"   Epochs {self.SUPCON_EPOCHS+1}-50: SSL (K=6/K=3 hard positives)")
    
    def set_epoch(self, epoch):
        """Called by training loop to update current epoch."""
        self.current_epoch = epoch
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clip_id = int(row['clip_id'])
        vessel_class = row['vessel_class']
        label = self.class_to_id[vessel_class]
        
        # Load anchor
        anchor = np.load(os.path.join(self.data_dir, f"tensor_{clip_id:06d}.npy")).astype(np.float32)
        
        # Select positive based on CURRENT EPOCH
        if self.current_epoch <= self.SUPCON_EPOCHS:
            # Phase 1: SupCon - random from same class
            candidates = [c for c in self.class_clip_ids[vessel_class] if c != clip_id]
            pos_clip_id = int(np.random.choice(candidates))
        else:
            # Phase 2: SSL - hard positive from pairing manifest
            if clip_id in self.pairing_lookup:
                pos_clip_id = int(np.random.choice(self.pairing_lookup[clip_id]))
            else:
                # Fallback to dynamic if missing
                candidates = [c for c in self.class_clip_ids[vessel_class] if c != clip_id]
                pos_clip_id = int(np.random.choice(candidates))
        
        positive = np.load(os.path.join(self.data_dir, f"tensor_{pos_clip_id:06d}.npy")).astype(np.float32)
        
        # Ensure shape [1, 80000]
        if anchor.ndim == 3:
            anchor = anchor.squeeze(0)
        if positive.ndim == 3:
            positive = positive.squeeze(0)
        
        return torch.from_numpy(anchor), torch.from_numpy(positive), label


# ============================================================================
# BARLOW TWINS LOSS
# ============================================================================

def barlow_twins_loss(z1, z2, lambd=BT_LAMBDA):
    """
    Barlow Twins loss - no inplace operations for gradient safety.
    """
    batch_size = z1.size(0)
    
    # Normalize
    z1_norm = (z1 - z1.mean(dim=0)) / (z1.std(dim=0) + 1e-6)
    z2_norm = (z2 - z2.mean(dim=0)) / (z2.std(dim=0) + 1e-6)
    
    # Cross-correlation matrix
    c = torch.mm(z1_norm.T, z2_norm) / batch_size
    
    # Loss
    on_diag = torch.diagonal(c).add(-1).pow(2).sum()
    off_diag = c.flatten()[:-1].view(c.size(0)-1, c.size(0)+1)[:, 1:].flatten().pow(2).sum()
    
    return on_diag + lambd * off_diag


# ============================================================================
# TRAINING WORKER (DDP with Gradient Accumulation)
# ============================================================================

def train_worker(rank, world_size, manifest_path, pairing_path, data_dir, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """
    DDP training worker with gradient accumulation and LR warmup.
    
    V3.2 uses:
    - batch_size=2 per GPU
    - gradient accumulation=8 steps
    - effective batch = 2 * 2 GPUs * 8 = 32
    - warmup for first 3 epochs
    """
    log_dir = "/kaggle/working/run_logs"
    logger = RankLogger(rank)
    
    try:
        # Initialize process group
        dist.init_process_group(
            backend='nccl',
            init_method='env://',
            world_size=world_size,
            rank=rank
        )
        torch.cuda.set_device(rank)
        logger.log(f"Process group initialized, device={rank}")
        
        # Model with SyncBatchNorm
        model = HybridSKEncoderV3(latent_dim=LATENT_DIM)
        model = nn.SyncBatchNorm.convert_sync_batchnorm(model)
        model = model.cuda(rank)
        model = DDP(model, device_ids=[rank])
        logger.log("Model ready (DDP + SyncBN)")
        
        # Dataset with distributed sampler
        dataset = HybridDataset(manifest_path, pairing_path, data_dir)
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=sampler,
            num_workers=2,
            pin_memory=True,
            drop_last=True
        )
        logger.log(f"DataLoader ready: {len(loader)} batches, effective batch={batch_size * world_size * GRAD_ACCUM_STEPS}")
        
        # Optimizer with warmup scheduler
        optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
        scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, epochs, BASE_LR)
        logger.log(f"Optimizer ready: LR={BASE_LR}, warmup={WARMUP_EPOCHS} epochs")
        
        # Loss history file (rank 0 only)
        if rank == 0:
            with open('/kaggle/working/loss_history.txt', 'w') as f:
                f.write("epoch,loss,lr\n")
        
        logger.log(f"Starting training: {epochs} epochs, grad_accum={GRAD_ACCUM_STEPS}")
        
        # Training loop with gradient accumulation
        for epoch in range(1, epochs + 1):
            sampler.set_epoch(epoch)
            dataset.set_epoch(epoch)  # Hybrid: switch SupCon→SSL at epoch 26
            model.train()
            total_loss = 0.0
            num_batches = 0
            
            # Update LR
            current_lr = scheduler.step()
            phase = "SupCon" if epoch <= 25 else "SSL"
            logger.log(f"Epoch {epoch}/{epochs} [{phase}] | LR: {current_lr:.2e}")
            
            optimizer.zero_grad(set_to_none=True)
            accum_loss = 0.0
            
            for step, (y1, y2, _) in enumerate(loader):
                y1 = y1.cuda(rank, non_blocking=True)
                y2 = y2.cuda(rank, non_blocking=True)
                
                # Forward pass
                emb1 = model(y1)
                emb2 = model(y2)
                loss = barlow_twins_loss(emb1, emb2, lambd=BT_LAMBDA)
                
                # Scale loss for gradient accumulation
                scaled_loss = loss / GRAD_ACCUM_STEPS
                scaled_loss.backward()
                
                accum_loss += loss.item()
                
                # Update weights every GRAD_ACCUM_STEPS
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    
                    total_loss += accum_loss / GRAD_ACCUM_STEPS
                    num_batches += 1
                    accum_loss = 0.0
                
                # Heartbeat every 50 steps
                if step % 50 == 0:
                    write_heartbeat(log_dir, epoch, step, loss.item(), rank)
            
            # Handle remaining gradients
            if accum_loss > 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                total_loss += accum_loss / GRAD_ACCUM_STEPS
                num_batches += 1
            
            avg_loss = total_loss / max(num_batches, 1)
            logger.log(f"Epoch {epoch}/{epochs} complete | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
            
            # Rank 0: logging and checkpoints
            if rank == 0:
                with open('/kaggle/working/loss_history.txt', 'a') as f:
                    f.write(f"{epoch},{avg_loss:.4f},{current_lr:.2e}\n")
                    f.flush()
                    os.fsync(f.fileno())
                
                # Checkpoints every 5 epochs (more frequent for debugging)
                if epoch % 5 == 0 or epoch == epochs:
                    save_path = f"/kaggle/working/BT_ckpt_epoch_{epoch:03d}.pth"
                    torch.save(
                        {"epoch": epoch, "encoder": model.module.state_dict()},
                        save_path
                    )
                    logger.log(f"Checkpoint saved: {save_path}")
        
        # Final model save (rank 0)
        if rank == 0:
            torch.save(model.module.state_dict(), "/kaggle/working/SKANN_SSL_V3_2_Final.pth")
            logger.log("✅ Training complete! Final weights saved.")
        
        dist.destroy_process_group()
        
    except Exception as e:
        log_crash(log_dir, e, rank)
        raise


def train_single_gpu(manifest_path, pairing_path, data_dir, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Fallback single-GPU training (no DDP)."""
    print(f"[SingleGPU] Starting V3.2 training: {epochs} epochs")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = HybridSKEncoderV3(latent_dim=LATENT_DIM).to(device)
    dataset = HybridDataset(manifest_path, pairing_path, data_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=True)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    scheduler = WarmupCosineScheduler(optimizer, WARMUP_EPOCHS, epochs, BASE_LR)
    
    with open('/kaggle/working/loss_history.txt', 'w') as f:
        f.write("epoch,loss,lr\n")
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        num_batches = 0
        
        current_lr = scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        
        for step, (y1, y2, _) in enumerate(loader):
            y1 = y1.to(device)
            y2 = y2.to(device)
            
            emb1 = model(y1)
            emb2 = model(y2)
            loss = barlow_twins_loss(emb1, emb2)
            
            scaled_loss = loss / GRAD_ACCUM_STEPS
            scaled_loss.backward()
            accum_loss += loss.item()
            
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                total_loss += accum_loss / GRAD_ACCUM_STEPS
                num_batches += 1
                accum_loss = 0.0
        
        if accum_loss > 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            total_loss += accum_loss / GRAD_ACCUM_STEPS
            num_batches += 1
        
        avg_loss = total_loss / max(num_batches, 1)
        print(f"[SingleGPU] Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f} | LR: {current_lr:.2e}")
        
        with open('/kaggle/working/loss_history.txt', 'a') as f:
            f.write(f"{epoch},{avg_loss:.4f},{current_lr:.2e}\n")
        
        if epoch % 5 == 0 or epoch == epochs:
            torch.save(
                {"epoch": epoch, "encoder": model.state_dict()},
                f"/kaggle/working/BT_ckpt_epoch_{epoch:03d}.pth"
            )
    
    torch.save(model.state_dict(), "/kaggle/working/SKANN_SSL_V3_2_Final.pth")
    print("[SingleGPU] ✅ Training complete!")


---
## Cell 5: Launch Training

In [ ]:
# Cell 5: Launch Training
import os
import random
import torch
import torch.multiprocessing as mp
from train_script_v3_2 import train_worker, train_single_gpu, log_crash
from train_script_v3_2 import EPOCHS, BATCH_SIZE, GRAD_ACCUM_STEPS, WARMUP_EPOCHS, BASE_LR

def launch():
    log("🚀 Launching SKANN-SSL V3.2.0 Training...")
    print("="*70)
    print("  V3.2.0 HYBRID — SupCon + SSL Combined")
    print("  ")
    print("  Input: 80,000 samples (5s @ 16kHz)")
    print("  Classes: small_craft, fishing_vessel, cargo_ship, tanker, no_vessel")
    print("  Total Clips: 12,000 (2,400 per class)")
    print("  ")
    print("  SK Kernels: (31, 63, 127, 255, 511, 1023)")
    print("  Projector: 512 → 4096 → 8192 → 128")
    print("  ")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Physical Batch: {BATCH_SIZE} per GPU")
    print(f"  Gradient Accumulation: {GRAD_ACCUM_STEPS} steps")
    print(f"  Effective Batch: {BATCH_SIZE * 2 * GRAD_ACCUM_STEPS}")
    print(f"  Learning Rate: {BASE_LR} with {WARMUP_EPOCHS}-epoch warmup")
    print("="*70)
    
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(random.randint(29500, 29999))
    
    torch.cuda.empty_cache()
    world_size = torch.cuda.device_count()
    log(f"GPUs detected: {world_size}")
    
    # V3.2 paths
    manifest = "/kaggle/input/skann-ssl-v3-2-dataset/master_dataset_manifest.csv"
    pairing = "/kaggle/input/skann-ssl-v3-2-dataset/pairing_manifest.csv"
    data_dir = "/kaggle/input/skann-ssl-v3-2-dataset/tensors/"
    log_dir = "/kaggle/working/run_logs"
    
    try:
        if world_size < 2:
            log("⚠️ Single GPU mode (no DDP/SyncBN)")
            train_single_gpu(manifest, pairing, data_dir, epochs=EPOCHS, batch_size=BATCH_SIZE)
        else:
            log(f"🚀 DDP mode with {world_size} GPUs + SyncBatchNorm")
            mp.spawn(
                train_worker,
                args=(world_size, manifest, pairing, data_dir, EPOCHS, BATCH_SIZE),
                nprocs=world_size,
                join=True
            )
        log("✅ Training complete!")
        
    except Exception as e:
        log_crash(log_dir, e, rank=-1)
        raise

launch()


---
## Cell 6: Extract Embeddings

In [ ]:
# Cell 6: Extract Embeddings
import torch
import numpy as np
import os
import glob
import json
import pandas as pd
from train_script_v3_2 import HybridSKEncoderV3

def extract_embeddings():
    log("🔬 Extracting embeddings...")
    
    # Find weights
    weights = "/kaggle/working/SKANN_SSL_V3_2_Final.pth"
    if not os.path.exists(weights):
        candidates = sorted(glob.glob("/kaggle/working/BT_ckpt_epoch_*.pth"))
        if not candidates:
            log("❌ No weights found!")
            return None, None
        weights = candidates[-1]
        log(f"Using checkpoint: {weights}")
    
    # Load model
    model = HybridSKEncoderV3(latent_dim=128)
    state = torch.load(weights, map_location='cpu')
    if 'encoder' in state:
        state = state['encoder']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval()
    model.cuda()
    log("Model loaded")
    
    # Load manifest
    manifest_path = "/kaggle/input/skann-ssl-v3-2-dataset/master_dataset_manifest.csv"
    tensor_dir = "/kaggle/input/skann-ssl-v3-2-dataset/tensors/"
    df = pd.read_csv(manifest_path)
    
    classes = sorted(df['vessel_class'].unique())
    class_to_id = {c: i for i, c in enumerate(classes)}
    
    embeddings = []
    labels = []
    clip_ids = []
    
    log(f"Processing {len(df)} clips...")
    
    with torch.no_grad():
        for idx, row in df.iterrows():
            clip_id = int(row['clip_id'])
            tensor_path = os.path.join(tensor_dir, f"tensor_{clip_id:06d}.npy")
            
            tensor = np.load(tensor_path).astype(np.float32)
            if tensor.ndim == 3:
                tensor = tensor.squeeze(0)
            x = torch.from_numpy(tensor).unsqueeze(0).cuda()
            
            emb = model(x).cpu().numpy().flatten()
            embeddings.append(emb)
            labels.append(class_to_id[row['vessel_class']])
            clip_ids.append(clip_id)
            
            if (idx + 1) % 2000 == 0:
                log(f"  Processed {idx + 1}/{len(df)}")
    
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    clip_ids = np.array(clip_ids)
    
    # Save
    np.save('/kaggle/working/embeddings_v3_2.npy', embeddings)
    np.save('/kaggle/working/labels_v3_2.npy', labels)
    np.save('/kaggle/working/clip_ids_v3_2.npy', clip_ids)
    
    with open('/kaggle/working/class_mapping_v3_2.json', 'w') as f:
        json.dump({'classes': classes, 'class_to_id': class_to_id}, f)
    
    log(f"✅ Saved embeddings: {embeddings.shape}")
    return embeddings, labels, classes

embeddings, labels, classes = extract_embeddings()

---
## Cell 7: Silhouette Analysis

In [ ]:
# Cell 7: Silhouette Analysis
import numpy as np
from sklearn.metrics import silhouette_score

if embeddings is not None:
    log("📊 Computing silhouette score...")
    
    # Cosine silhouette (matches V2.1.0)
    silhouette = silhouette_score(embeddings, labels, metric='cosine')
    
    print("\n" + "="*50)
    print(f"  SILHOUETTE SCORE (cosine): {silhouette:.4f}")
    print("="*50)
    
    # Compare to baselines
    print("\n📈 Comparison:")
    print(f"   V1 Baseline:   0.3997")
    print(f"   V2.1.0:        0.8299")
    print(f"   V3.1 (failed): ~0.2-0.3 (physics bugs)")
    print(f"   V3.2 (this):   {silhouette:.4f}")
    
    if silhouette > 0.8:
        print("\n🎉 EXCELLENT: Matches or exceeds V2.1.0!")
    elif silhouette > 0.6:
        print("\n✅ GOOD: Significant improvement over V1")
    elif silhouette > 0.4:
        print("\n⚠️ MODERATE: Room for improvement")
    else:
        print("\n❌ LOW: Check for issues")
    
    with open('/kaggle/working/silhouette.txt', 'w') as f:
        f.write(str(silhouette))
    
    log(f"✅ Silhouette score saved: {silhouette:.4f}")

---
## Cell 8: Silhouette Distribution Plot

In [ ]:
# Cell 8: Silhouette Distribution Plot
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples

if embeddings is not None:
    log("📊 Generating silhouette distribution plot...")
    
    sample_silhouettes = silhouette_samples(embeddings, labels, metric='cosine')
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    y_lower = 10
    colors = plt.cm.Set1(np.linspace(0, 1, len(classes)))
    
    for i, cls in enumerate(classes):
        mask = labels == i
        cls_silhouettes = sample_silhouettes[mask]
        cls_silhouettes.sort()
        
        size = len(cls_silhouettes)
        y_upper = y_lower + size
        
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cls_silhouettes,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, cls, fontsize=10)
        
        y_lower = y_upper + 10
    
    ax.axvline(x=silhouette, color='red', linestyle='--', label=f'Mean: {silhouette:.4f}')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster (Class)')
    ax.set_title('SKANN-SSL V3.2.0 Silhouette Distribution\n(5 classes, 12K clips, physics fixed)')
    ax.legend(loc='best')
    ax.set_xlim(-0.2, 1.0)
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/silhouette_distribution_v3_2.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    log("✅ Silhouette plot saved")

---
## Cell 9: UMAP Visualization

In [ ]:
# Cell 9: UMAP Visualization
import matplotlib.pyplot as plt
import umap
import json
import os

# Reload if needed
if 'embeddings' not in globals() or embeddings is None:
    embeddings = np.load('/kaggle/working/embeddings_v3_2.npy')
    labels = np.load('/kaggle/working/labels_v3_2.npy')
    with open('/kaggle/working/class_mapping_v3_2.json', 'r') as f:
        class_info = json.load(f)
        classes = class_info['classes']

if 'silhouette' not in globals():
    try:
        silhouette = float(open('/kaggle/working/silhouette.txt').read().strip())
    except:
        silhouette = None

if embeddings is not None:
    log("📊 Computing UMAP projection...")
    
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=42
    )
    embedding_2d = reducer.fit_transform(embeddings)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    colors = plt.cm.Set1(np.linspace(0, 1, len(classes)))
    
    for i, cls in enumerate(classes):
        mask = labels == i
        ax.scatter(
            embedding_2d[mask, 0],
            embedding_2d[mask, 1],
            c=[colors[i]],
            label=f'{cls} ({mask.sum()})',
            alpha=0.6,
            s=20
        )
    
    sil_str = f'{silhouette:.4f}' if silhouette else 'N/A'
    ax.set_title(f'SKANN-SSL V3.2.0 Embedding Space\nSilhouette: {sil_str} (5 classes, 12K clips)', fontsize=14)
    ax.legend(loc='best', fontsize=10)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/umap_v3_2.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    log("✅ UMAP plot saved")
    
    print("\n📈 Per-Class Distribution:")
    for i, cls in enumerate(classes):
        count = np.sum(labels == i)
        print(f"   {cls}: {count} samples")

---
## Cell 10: Export Production Bundle

In [ ]:
# Cell 10: Export Production Bundle
import torch
import numpy as np
import joblib
import json
import os
import glob
from datetime import datetime
from train_script_v3_2 import HybridSKEncoderV3, EPOCHS, BATCH_SIZE, GRAD_ACCUM_STEPS, BASE_LR

def export_production_bundle():
    log("📦 Exporting Production Bundle...")
    
    weights = "/kaggle/working/SKANN_SSL_V3_2_Final.pth"
    if not os.path.exists(weights):
        candidates = sorted(glob.glob("/kaggle/working/BT_ckpt_epoch_*.pth"))
        if not candidates:
            log("❌ No weights found!")
            return
        weights = candidates[-1]
    
    model = HybridSKEncoderV3(latent_dim=128)
    state = torch.load(weights, map_location='cpu')
    if 'encoder' in state:
        state = state['encoder']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    
    embeddings = np.load("/kaggle/working/embeddings_v3_2.npy")
    labels = np.load("/kaggle/working/labels_v3_2.npy")
    clip_ids = np.load("/kaggle/working/clip_ids_v3_2.npy")
    
    with open("/kaggle/working/class_mapping_v3_2.json", 'r') as f:
        class_info = json.load(f)
    
    classes = class_info['classes']
    centroids = {}
    for i, cls in enumerate(classes):
        mask = labels == i
        centroids[cls] = embeddings[mask].mean(axis=0)
    
    try:
        silhouette = float(open('/kaggle/working/silhouette.txt').read().strip())
    except:
        silhouette = None
    
    bundle = {
        'version': 'V3.2.0',
        'created': datetime.now().isoformat(),
        'model_state_dict': model.state_dict(),
        'model_config': {
            'latent_dim': 128,
            'input_samples': 80000,
            'sample_rate': 16000,
            'duration_sec': 5.0,
            'sk_kernel_sizes': (31, 63, 127, 255, 511, 1023),
            'num_classes': 5
        },
        'classes': classes,
        'class_to_id': class_info['class_to_id'],
        'centroids': centroids,
        'embeddings': embeddings,
        'labels': labels,
        'clip_ids': clip_ids,
        'silhouette_score': silhouette,
        'training_info': {
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'gradient_accumulation': GRAD_ACCUM_STEPS,
            'effective_batch': BATCH_SIZE * 2 * GRAD_ACCUM_STEPS,
            'optimizer': 'AdamW',
            'lr': BASE_LR,
            'weight_decay': 0.01,
            'loss': 'Barlow Twins',
            'lambda': 5e-3,
            'warmup_epochs': 3,
            'total_clips': 12000,
            'physics_fixes': ['2Hz_swell_artifact', 'resonance_coverage']
        }
    }
    
    bundle_path = "/kaggle/working/SKANN_SSL_V3_2_Production_Bundle.joblib"
    joblib.dump(bundle, bundle_path)
    
    size_mb = os.path.getsize(bundle_path) / (1024 * 1024)
    log(f"✅ Bundle saved: {bundle_path} ({size_mb:.1f} MB)")
    
    print("\n" + "="*50)
    print("📦 PRODUCTION BUNDLE CONTENTS")
    print("="*50)
    print(f"   Version: {bundle['version']}")
    print(f"   Classes: {bundle['classes']}")
    print(f"   Embeddings: {bundle['embeddings'].shape}")
    print(f"   Silhouette: {bundle['silhouette_score']}")
    print(f"   Input: {bundle['model_config']['input_samples']} samples ({bundle['model_config']['duration_sec']}s)")
    
    return bundle_path

bundle_path = export_production_bundle()

---
## Cell 11: Summary & Next Steps

In [ ]:
# Cell 11: Summary
import os

print("\n" + "="*70)
print("🏁 SKANN-SSL V3.2.0 TRAINING COMPLETE")
print("="*70)

outputs = [
    "/kaggle/working/SKANN_SSL_V3_2_Final.pth",
    "/kaggle/working/SKANN_SSL_V3_2_Production_Bundle.joblib",
    "/kaggle/working/embeddings_v3_2.npy",
    "/kaggle/working/labels_v3_2.npy",
    "/kaggle/working/umap_v3_2.png",
    "/kaggle/working/silhouette_distribution_v3_2.png",
    "/kaggle/working/loss_history.txt"
]

print("\n📁 OUTPUT FILES:")
for f in outputs:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)
        print(f"   ✅ {os.path.basename(f)} ({size:.1f} MB)")
    else:
        print(f"   ❌ {os.path.basename(f)} (not found)")

print("\n📋 V3.2 KEY IMPROVEMENTS:")
print("   ✅ 12,000 clips (5× more than V3.1)")
print("   ✅ Physics bugs fixed (2Hz artifact resolved)")
print("   ✅ Gradient accumulation for stable training")
print("   ✅ Learning rate warmup")

print("\n📋 NEXT STEPS:")
print("   1. Download SKANN_SSL_V3_2_Production_Bundle.joblib")
print("   2. Upload to Google Drive for Stage 6/7")
print("   3. Update Stage 6 evaluation for 5 classes")
print("   4. Generate territory map and confusion matrix")
print("   5. Compare silhouette with V2.1.0 baseline (0.8299)")

print("\n" + "="*70)
print("🎉 V3.2.0 READY FOR DEPLOYMENT")
print("="*70)